In [ ]:
import pandas as pd
import geopandas as gpd
from src.config import *
from src.funcs import *
import re

In [ ]:
settlements_chosen=gpd.read_file(DIR_META/"Benin Sampling Surveyed.gpkg")
data=pd.read_csv(DIR_MAPPED_DATA)

# "Lalo" is duplicated in the gpkg (two placemarks, effectively identical geometry/attributes),
# which double-matches respondents in the spatial join below. Drop the duplicate.
settlements_chosen = settlements_chosen.drop_duplicates(subset='Name', keep='first')

# One cluster is literally named "NA". pandas' default read_csv NA-value list treats the text
# "NA" as missing, so every time this survives a CSV round-trip it silently becomes NaN
# (this nulled out 'location' for the ~24 respondents in that cluster). Rename it so it's
# unambiguous through any future to_csv/read_csv cycle.
settlements_chosen['Name'] = settlements_chosen['Name'].replace('NA', 'Unnamed Cluster (NA)')


In [ ]:
# Convert survey data to geodataframe
gdf_data = gpd.GeoDataFrame(data, geometry = gpd.points_from_xy(data['gps-Longitude'], data['gps-Latitude']))\
                .set_crs("epsg:4326")

In [ ]:
# Reproject to match CRS
gdf_data = gdf_data.to_crs(settlements_chosen.crs)

Spatially join where the survey took place and the survey bounds

In [ ]:
data_joined = gdf_data.sjoin(settlements_chosen,predicate='intersects')

print(len(data), len(data_joined))

In [ ]:
## Snap respondents that fall just outside every cluster polygon (GPS drift / polygon
## edge effects) but are within 1km of one, onto the nearest cluster. Anyone still
## outside every cluster beyond that gets exported for manual review instead of dropped.
SNAP_DISTANCE_M = 1000
UTM_CRS = "epsg:32631"  # UTM 31N -- covers Benin, gives us metres for the distance check

matched_keys = set(data_joined['KEY'])
unmatched = gdf_data.loc[~gdf_data['KEY'].isin(matched_keys)].copy()
print(f"{len(unmatched)} respondents fall outside every cluster polygon before snapping")

unmatched_m = unmatched.to_crs(UTM_CRS)
chosen_m = settlements_chosen.to_crs(UTM_CRS)

snap_rows = []
still_outside_idx = []
for idx, row in unmatched_m.iterrows():
    dists = chosen_m.geometry.distance(row.geometry)
    nearest_i = dists.idxmin()
    nearest_dist = dists.loc[nearest_i]
    if nearest_dist <= SNAP_DISTANCE_M:
        snap_rows.append({'index': idx, 'Name': chosen_m.loc[nearest_i, 'Name'], 'snap_distance_m': nearest_dist})
    else:
        still_outside_idx.append(idx)

print(f"Snapped {len(snap_rows)} respondents onto the nearest cluster (within {SNAP_DISTANCE_M}m)")
print(f"{len(still_outside_idx)} respondents remain outside every cluster even with the {SNAP_DISTANCE_M}m buffer")

if snap_rows:
    snap_df = pd.DataFrame(snap_rows).set_index('index')
    snapped = gdf_data.loc[snap_df.index].copy()
    snapped['Name'] = snap_df['Name']
    print("\nSnapped respondents:")
    print(snap_df)
    data_joined = pd.concat([data_joined, snapped], ignore_index=False)

# Export respondents still outside every cluster (even after the 1km snap) for manual review
outside_export = gdf_data.loc[still_outside_idx].drop(columns='geometry')
outside_path = DIR_VALIDATE / "respondents_outside_all_clusters.csv"
outside_export.to_csv(outside_path, index=False)
print(f"\nExported {len(outside_export)} still-outside respondents to {outside_path}")

In [ ]:
## Manual review (2026-08-19): of the respondents still outside every cluster after the
## 1km snap, the user manually checked all 8 against the "Toffo" cluster boundary:
##   - 6 of 8 are genuinely in Toffo -- GPS is accurate (all cluster within ~50m of each
##     other, near 6.8444437, 2.1028666), just outside the drawn polygon boundary.
##   - 2 of 8 (same enumerator, sandrakoukponou@gmail.com, submissions on 2026-07-16 and
##     2026-07-18) had what the user believes was a bad SurveyCTO GPS reading -- their
##     recorded location text ("Gboa...", "Awonou...") was wrong too, not just a polygon
##     edge case. Their coordinates are replaced with the Toffo reference point.
## All 8 are assigned to Toffo; only the 2 bad-GPS ones get their coordinates overwritten.
TOFFO_REF_LAT, TOFFO_REF_LON = 6.8444437, 2.1028666
BAD_GPS_KEYS = [
    "uuid:e2d73352-7dd9-4e8b-a848-cbbc900cdde1",  # 2026-07-16, typed as "Gboa..."
    "uuid:2cf5a65e-e58d-417c-add6-ab94813d9012",  # 2026-07-18, typed as "Awonou..."
]
MANUAL_TOFFO_KEYS = BAD_GPS_KEYS + [
    "uuid:cb40df61-c78d-41cc-b8a7-bd57f3d92531",
    "uuid:9f1e35a9-1276-4eb3-a487-7b44300a6e58",
    "uuid:4ab53aed-7c95-41be-b3a9-3216e7aa88a3",
    "uuid:41beb929-2008-4040-9e3c-2d3d87ede9bf",
    "uuid:7ea0e467-6eb2-4324-b992-2282ab85f710",
    "uuid:7a6b88ae-dcf6-4d04-80f2-76b865bc150b",
]

manual_idx = gdf_data.loc[gdf_data['KEY'].isin(MANUAL_TOFFO_KEYS)].index.tolist()
print(f"Manually assigning {len(manual_idx)} respondents to Toffo (expected 8)")

manual_rows = gdf_data.loc[manual_idx].copy()
bad_gps_mask = manual_rows['KEY'].isin(BAD_GPS_KEYS)
manual_rows.loc[bad_gps_mask, 'gps-Latitude']  = TOFFO_REF_LAT
manual_rows.loc[bad_gps_mask, 'gps-Longitude'] = TOFFO_REF_LON
manual_rows['geometry'] = gpd.GeoSeries(
    gpd.points_from_xy(manual_rows['gps-Longitude'], manual_rows['gps-Latitude']), crs="epsg:4326"
).to_crs(settlements_chosen.crs).values
manual_rows['Name'] = 'Toffo'

# Remove these from the "still outside" bucket before it gets exported, and fold them
# into the matched set so they flow through to the final weighted dataset.
still_outside_idx = [i for i in still_outside_idx if i not in manual_idx]
data_joined = pd.concat([data_joined, manual_rows], ignore_index=False)

Drop duplicate assignments

In [ ]:
data_joined_dedupe = data_joined.drop_duplicates(keep='first')

Examine where there are discrepancies in settlement names:

In [ ]:
keep_cols = data.columns.tolist() + ['Name', 'geometry']
data_joined_dedupe = data_joined_dedupe[keep_cols]


data_joined_dedupe.loc[data_joined_dedupe.Name != data_joined_dedupe.location].groupby(['Name', 'location']).count()

Correct the locations for surveys that spatially took place in another settlement: 

In [ ]:
data_joined_dedupe.loc[(data_joined_dedupe.Name =="Doga; Agonlin-Houégbo") &
                     (data_joined_dedupe.location == "Zassa; Hlanhonou; TohouË; Gnanli"), 'location'] = "Doga; Agonlin-Houégbo"



In [ ]:
data_joined_dedupe.loc[(data_joined_dedupe.Name =="Toffo") &
                     (data_joined_dedupe.location == "HounnouviÈ; SËdji; ZËko; Ahito; Ahita; Gomey"), 'location'] = 'Toffo'

In [ ]:
# Every remaining location-vs-Name mismatch (435/827 respondents, checked 2026-08-19) is an
# encoding/formatting artifact, not a genuine disagreement about where the respondent was:
# the QA file's free-text 'location name' column has widespread mojibake (e.g. "TohouË" for
# "Tohouè", "È"/"Ë"/"Ù" corruption throughout) plus inconsistent capitalization ("AYOKPO" vs
# "Ayokpo") and abbreviation vs. the gpkg's clean 'Name' field. Verified every mismatch group
# is 1:1 (one Name <-> one location variant) -- i.e. always the same physical cluster, never
# a real relocation. 'Name' comes from the GPS spatial join against the same gpkg used
# everywhere downstream (settlements_chosen), so it's the canonical label; the weighting
# merge keys off 'location', so without this fix over half the respondents silently fail to
# match their cluster and get NaN weights.
mismatch_before = (data_joined_dedupe['location'] != data_joined_dedupe['Name']).sum()
data_joined_dedupe['location'] = data_joined_dedupe['Name']
print(f"Overwrote 'location' with the spatially-joined 'Name' for {mismatch_before} respondents "
      f"whose QA-typed location text didn't match byte-for-byte (encoding/formatting only)")

Export with corrections

In [ ]:
len(data_joined_dedupe)

In [ ]:
data_joined_dedupe.to_csv(DIR_MAPPED_DATA)